In [ ]:
import os 
from IPython.display import display
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell  
from ipywidgets import interact  
InteractiveShell.ast_node_interactivity = "all"   # type: ignore
import ipywidgets as widgets
from ipywidgets import interact 
from pandas import Index
from typing import Any, TypedDict
import torch
from src.base.lit_module.components.metrics_standard import MyMultiClassAccuracy

In [ ]:
""" 
Required classes. 
"""

class MetricMetadata(TypedDict):
    epochs: list[float]
    steps: list[float]

class PerStepMetricInfo(TypedDict):
    values: np.ndarray
    save_path: Path
    header: list[str]

In [ ]:
"""
Interactive analysis functionality.
"""

# def show_col_summary(column: Any, frame):
#     print(frame[column].describe())

# def show_summary(dfs: list[pd.DataFrame], v_dir: Path):
#     print(f"Results aggregated over {v_dir.stem} in {v_dir.parent.parent.stem}/{v_dir.parent.stem}")
#     for frame in dfs:
#         interact(show_col_summary, frame=frame, column=list(frame.columns))
#         frame 


In [ ]:
""" 
Visualization functionality. 
"""

def viz_sep_col(metric: PerStepMetricInfo, save_fig: bool = True, save_derived_metric: bool = False) -> None:
    """
    For a metric object instance, visualize each col of `values` with its associated string in `header`.
    Internal output is figure with plots per column, header label combo.
    Optionally save each to compressed numpy file with flags (NOT recommended).
    Optionally save figure (Recommended). 
    TODO: But with what path, might as well maybe create separate instance separate from global df, not really. 
    """
    n_cols = metric["values"].shape[1]
    fig, axs = plt.subplots(1, n_cols, sharex=True, sharey=True, figsize=(4 * n_cols, 4))
    fig.subplots_adjust(wspace=0, hspace=0)

    for i, _ in enumerate(metric["header"]):
        col_vals = metric["values"][:,i]
        col_name = metric["header"][i]
        clean_header = col_name.replace('.', '_')
        col_path = Path(metric['save_path'].parent.resolve() / clean_header)
        axs[i].plot(
            col_vals, 
            # linestyle="dashed", 
            # color="black", 
            # marker="o", 
            label=clean_header
        )
        axs[i].legend()

        if save_derived_metric:
            if not Path.exists(col_path):
                col_path.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(file=col_path.resolve(), derived_metric = col_vals)

    if save_fig: 
        save_path = Path(str(metric['save_path'].resolve()) + ".png")
        fig.savefig(save_path, format="png")
        
def viz_all_col(metric_df: pd.DataFrame, unified_name: str = "Train Accuracy") -> None:
    """
    For a metric object instance, visualize all col of `values` with its associated string in `header`.
    Internal output is figure with 1 plot for all column , labelled by col name in Index.
    Optionally save as compressed numpy file with flags per viz (Recommended).
    Optionally save figure (Recommended) per viz. 
    """
    n_cols = len(metric_df.columns)
    fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(n_cols, n_cols))
    for col in metric_df.columns:
        # display(list(metric_df.index.get_level_values(1)))
        # display(metric_df.loc[:, col].describe())
        # display(metric_df.loc[:, col].max())
        line, = ax.plot(list(metric_df.index.get_level_values(1)), list(metric_df.loc[:, col].values))
        line.set_label(col)
    ax.set_title("Accuracy")
    # ax.set_xticklabels(list(metric_df.index.get_level_values(1)))
    ax.set_ylabel(unified_name)
    ax.legend()
    plt.show()

In [ ]:
"""
Standard metric analysis helpers. 
"""
def summarize_run_acc(run_metrics_paths: list[Path], prefix:str):
    """ 
    This summarises accuracy metrics across a run.
    Output is in the parent directory of the current path.  
    """
    enumerated_metrics = enumerate(run_metrics_paths)

    run_dfs:list[pd.DataFrame] = []
    sum_df: pd.DataFrame
    for idx, train_metrics_path in enumerated_metrics:
        metric = torch.load(train_metrics_path.resolve(), map_location="cpu")

        classes = [k for k in list(metric.keys()) if k != "epochs" and k != "steps"] # Do not add epochs, steps back into classes, they get merged into expected keys, and their state is reloaded when seen as part of labels, as key names, then again as metadata frmo load state dict method

        base_acc = MyMultiClassAccuracy(classes) 
        base_acc.load_state_dict(metric)

        # Convert tensors not addressed by `.compute()`
        clean_epochs = [tensor.item() for tensor in base_acc.metric_state["epochs"]]
        clean_steps = [tensor.item() for tensor in base_acc.metric_state["steps"]]

        metadata = MetricMetadata(epochs=clean_epochs, steps=clean_steps) # Unify values in new dict
        metadata_df = pd.DataFrame(metadata)
        head = metadata_df.head(10)
        index = pd.MultiIndex.from_frame(metadata_df)

        computed = base_acc.compute()

        data_df = pd.DataFrame(computed, index=index).replace(100, 0.2, inplace=True)
        if idx == 0:
            sum_df = pd.DataFrame(data=np.zeros(data_df.shape), index=data_df.index, columns=data_df.columns)
            sum_sq_df = pd.DataFrame(data=np.zeros(data_df.shape), index=data_df.index, columns=data_df.columns)
        
        sum_df = sum_df + data_df # type: ignore
        sum_sq_df = sum_sq_df + (data_df ** 2) # type: ignore

    num_metrics = idx + 1 # type: ignore
    global_mean_df = sum_df / (idx + 1) # type: ignore
    assert type(global_mean_df) is pd.DataFrame
    global_std_df = (sum_sq_df / num_metrics) - (global_mean_df ** 2) # type: ignore
    assert type(global_std_df) is pd.DataFrame
    summary_mean = global_mean_df.describe()
    summary_std = global_std_df.describe() 
    display(summary_mean)
    display(summary_std)

    viz_all_col(global_mean_df, unified_name="Train Accuracy Mean")    
    viz_all_col(global_std_df, unified_name="Train Accuracy Std")
    print("shown")    

In [ ]:
""" 
GCS analysis helpers. 
"""
def analyse_cs_metrics(metrics_paths: list[Path], prefix:str) -> None:
    """ 
    Perform analysis steps of correlation store metrics. 
    """
    summarize_run_acc(metrics_paths, prefix)

In [ ]:
outputs_dir = Path().resolve() / "outputs"
corr_store_names = dict(normed="GCS", raw="RGCS")

In [ ]:
""" 
Main entry point for analysis. 
"""

for experim_dir in sorted(outputs_dir.iterdir()):
    version_dirs = sorted(experim_dir.glob("multijob/R*"))

    for v_dir in version_dirs:
        print("Train accuracy run metrics for", v_dir)

        # Get all csv file path matches to default logger file name for standard metrics
        # train_acc_paths = sorted(v_dir.rglob("run*/train_acc.pt", case_sensitive=False))
        # if len(train_acc_paths) > 0:
        #     summarize_run_acc(train_acc_paths, prefix=train_acc_paths[0].stem)

        print("Val accuracy run metrics for", v_dir)

        val_acc_paths = sorted(v_dir.rglob("run*/val_acc.pt", case_sensitive=False))
        if len(val_acc_paths) > 0:
            summarize_run_acc(val_acc_paths, prefix=val_acc_paths[0].stem)

        # Get all csv file path matches to custom name for both cs metrics
        # gcs_paths = sorted(v_dir.rglob(f"run*/train/*{corr_store_names["normed"]}*.pt", case_sensitive=False))
        # analyse_cs_metrics(gcs_paths, "gcs")

        # rgcs_paths = sorted(v_dir.rglob(f"run*/train/*{corr_store_names["raw"]}*.pt", case_sensitive=False))
        # analyse_cs_metrics(rgcs_paths, "rgcs")